# A-Maze-ing

## Read the configuration file

Just open the file, split the options turn them into a dictionary.

In [5]:
with open("config.txt") as f:
    txt = f.read()
    opts = txt.split("\n")
    tup = [tuple(i.split("=")) for i in opts]
    conf = {i[0]: i[1] for i in tup}
print(conf)

{'WIDTH': '9', 'HEIGHT': '7', 'ENTRY': '0,0', 'EXIT': '7,5', 'OUTPUT_FILE': 'maze.txt', 'PERFECT': 'True'}


## Create a $h \times w$ array

Two different arrays:

* The maze itself.
* The visited cells.

The 42 was hardcoded.

In [6]:
from numpy import zeros, ones, mean, array


size = (int(conf["HEIGHT"]), int(conf["WIDTH"]))
maze = ones(size, int) * 15
visited = zeros(size, int)
if visited.shape[0] < 7 and visited.shape[1] < 9:
    print("The '42' cannot be printed in a maze this size.")
else:
    hs, ws = visited.shape[0] // 2 - 2, visited.shape[1] // 2 - 3
    pos_42 = (
        (hs, ws), (hs + 1, ws), (hs + 2, ws), (hs + 2, ws + 1),
        (hs + 2, ws + 2), (hs + 3, ws + 2), (hs + 4, ws + 2),
        (hs, ws + 4), (hs, ws + 5), (hs, ws + 6), (hs + 1, ws + 6),
        (hs + 2, ws + 6), (hs + 2, ws + 5), (hs + 2, ws + 4),
        (hs + 3, ws + 4), (hs + 4, ws + 4), (hs + 4, ws + 5),
        (hs + 4, ws + 6)
    )
    for i in pos_42:
        visited[i[0]][i[1]] = -1
    lab_42 = [
        (hs + 1, ws + 3),
        (hs + 1, ws + 4),
        (hs + 1, ws + 5),
        (hs + 3, ws + 5),
        (hs + 3, ws + 6),
        (hs + 3, ws + 7),
    ]
    perfect = conf["PERFECT"]
    if perfect is False:
        lab_42.extend(
            [
                (hs + 2, ws + 3),
                (hs + 3, ws + 3),
                (hs + 4, ws + 3),
                (hs + 5, ws + 3)
            ]
        )
    for i in lab_42:
        visited[i[0], i[1]] = 1
print(visited)"""
maze[hs + 1][ws + 3] = 13
maze[hs + 1][ws + 4] = 5
maze[hs + 1][ws + 5] = 7
maze[hs + 3][ws + 5] = 13
maze[hs + 3][ws + 6] = 5
maze[hs + 3][ws + 7] = 7
if perfect is False:
    maze[hs + 1][ws + 3] = 8
    maze[hs + 2][ws + 3] = 10
    maze[hs + 3][ws + 3] = 10
    maze[hs + 4][ws + 3] = 10
    maze[hs + 5][ws + 3] = 14"""
for i in maze:
    print(i)

[[ 0  0  0  0  0  0  0  0  0]
 [ 0 -1  0  0  0 -1 -1 -1  0]
 [ 0 -1  0  0  1  1  1 -1  0]
 [ 0 -1 -1 -1  0 -1 -1 -1  0]
 [ 0  0  0 -1  0 -1  1  1  1]
 [ 0  0  0 -1  0 -1 -1 -1  0]
 [ 0  0  0  0  0  0  0  0  0]]
[15 15 15 15 15 15 15 15 15]
[15 15 15 15 15 15 15 15 15]
[15 15 15 15 13  5  7 15 15]
[15 15 15 15 15 15 15 15 15]
[15 15 15 15 15 15 13  5  7]
[15 15 15 15 15 15 15 15 15]
[15 15 15 15 15 15 15 15 15]


## Manage errors

Checks:

* Maze dimensions must be positive.
* Maze entrance and exit are inside the maze.
* Maze entrance and exit are not inside forbidden tiles.
* All maze walls in a tile are coherent with the neighbouring tiles.

In [3]:
class MazeError(Exception):
    def __init__(self, msg: str):
        self.msg = msg


start = conf["ENTRY"].split(",")
end = conf["EXIT"].split(",")
entr = (int(start[0]), int(start[1]))
exit = (int(end[0]), int(end[1]))

if size[0] <= 0 or size[1] <= 0:
    raise MazeError("Maze dimensions must be positive")

in_bounds = [
    entr[0] >= 0,
    entr[1] >= 0,
    entr[0] < size[0],
    entr[1] < size[1],
    exit[0] >= 0,
    exit[1] >= 0,
    exit[0] < size[0],
    exit[1] < size[1]
]
if False in in_bounds:
    raise MazeError("Entrance or exit of the maze seem to be out of bounds")

if visited.shape[0] > 6 or visited.shape[1] > 8:
    in_42 = []
    in_42.extend([True for i in pos_42 if i[0] == entr[0] and i[1] == entr[1]])
    in_42.extend([True for i in pos_42 if i[0] == exit[0] and i[1] == exit[1]])
    if True in in_42:
        raise MazeError("Entrance or exit of the maze seem to be inside the 42")

i, j = 0, 0

if i in (0, size[0] - 1) or j in (0, size[1] - 1):
    if i == 0 and maze[i][j] % 2 == 0:
        raise MazeError(f"Maze tile {(i, j)} has incoherent walls")
    if i == size[0] and maze[i][j] // 4 == 0:
        raise MazeError(f"Maze tile {(i, j)} has incoherent walls")
    if j == 0 and maze[i][j] // 8 == 0:
        raise MazeError(f"Maze tile {(i, j)} has incoherent walls")
    if j == size[1] and maze[i][j] // 2 == 0:
        raise MazeError(f"Maze tile {(i, j)} has incoherent walls")
else:
    if maze[i][j] % 2 != maze[i - 1][j] // 4:
        raise MazeError(f"Maze tile {(i, j)} has incoherent walls")
    if maze[i][j] // 4 != maze[i + 1][j] % 2:
        raise MazeError(f"Maze tile {(i, j)} has incoherent walls")
    if maze[i][j] // 8 != maze[i][j - 1] // 2:
        raise MazeError(f"Maze tile {(i, j)} has incoherent walls")
    if maze[i][j] // 2 != maze[i][j + 1] // 8:
        raise MazeError(f"Maze tile {(i, j)} has incoherent walls")

## Maze generator

Gonna be a Wilson algorithm.

In [ ]:
import random as rng


def manhattan(curr: tuple[int], neigh: tuple[int]) -> int:
    return abs(curr[0] - neigh[0]) + abs(curr[1] - neigh[1])



def connect(maze: array[int], curr: tuple[int], neigh: tuple[int]) -> None:
    if manhattan(curr, neigh) != 0:
        return


def direction() -> tuple[str, int]:
    if rng.random() < .5:
        axis = "V"
        if rng.random() < .5:
            step = -1
        else:
            step = 1
    else:
        axis = "H"
        if rng.random() < .5:
            step = -1
        else:
            step = 1
    return (axis, step)


def count_visited(visited: array[int]) -> int:
    n = 0
    for i in visited.shape[0]:
        for j in visited.shape[1]:
            if visited[i][j] == 1:
                n += 1
    return n

    
    if curr[0] == neigh[0] + 1:
        maze[curr[0]][curr[1]] -= 1
        maze[neigh[0]][neigh[1]] -= 4
    if curr[0] == neigh[0] - 1:
        maze[curr[0]][curr[1]] -= 4
        maze[neigh[0]][neigh[1]] -= 1
    if curr[1] == neigh[1] - 1:
        maze[curr[0]][curr[1]] -= 2
        maze[neigh[0]][neigh[1]] -= 8
    if curr[1] == neigh[1] + 1:
        maze[curr[0]][curr[1]] -= 2
        maze[neigh[0]][neigh[1]] -= 8


# Initialization
current = [rng.randrange(0, size[0] - 1), rng.randrange(0, size[1] - 1)]
visited[current[0]][current[1]] = 1
# Alodus-Broder Algorithm
while count_visited(visited) < maze.shape[0] * maze.shape[1] // 3 - 18:
    direct = direction()
    try:
        if direct[0] == "V":
            neighbour = [current[0] + direct[1], current[1]]
        else:
            neighbour = [current[0], current[1] + direct[1]]
    except IndexError:
        continue
    if visited[neighbour[0], neighbour[1]] == -1:
        continue
    elif visited[neighbour[0], neighbour[1]] == 0:
        connect(maze, tuple(current), tuple(neighbour))
    current = neighbour.copy()
# Wilson Algorithm
yet = set()
for i in maze.shape[0]:
    for j in maze.shape[1]:
        if yet[i, j] == 0:
            yet.add((i, j))
while len(yet) > 0:
    path = []
    current = rng.choice(yet)
    visited[current[0], current[1]] = 2
    while current in yet:
        path.append(current)
        if path.count(current) > 1:
            visited[current[0], current[1]] = 0
            path.pop()
            while path[-1] != current:
                visited[current[0], current[1]] = 0
                path.pop()
        direct = direction()
        if last_direction is None or last_direction != direction:

## Maze solution

Gonna be a recursive backtracker.

Better: A\*:

```python
# G cost: distance from starting node
# H (heuristic) cost: distance from end node
# h_cost = abs(x_1 - x_2) + abs(y_1 - y_2)
# F cost: G cost + H cost
# open: set of nodes to be evaluated
# closed: set of already evaluated nodes

while True:
    current = "node in open with lowest f_cost"
    remove(current, open)
    add(current, closed)
    
    if current == exit:
        return
    
    for i in neighbours(current):
        if accessible(i) == 0 or i in closed:
            continue

        if f_cost(i) < f_cost(current) or i not in open:
            set_f_cost(i)
            current = parent(i)
            if i not in open:
                add(i, open)
```

## Maze output

In [ ]:
txt = ""
with open(conf['OUTPUT_FILE'], 'w') as out:
    for i in range(size[0]):
        for j in range(size[1]):
            out.write(hex(maze[i][j]))
        out.write("\n")

    out.write("\n")
    out.write(f"{entr}          # entry (x,y)\n")
    out.write(f"{exit}          # exit (x,y)\n")
    # out.write(sol)